In [ ]:
# =====================================
# INSTALL REQUIRED LIBRARIES
# =====================================

!pip install -q groq PyPDF2

# =====================================
# IMPORT LIBRARIES
# =====================================

import PyPDF2
from groq import Groq

# =====================================
# ENTER YOUR GROQ API KEY
# =====================================

GROQ_API_KEY = "you_api_key"

client = Groq(api_key=GROQ_API_KEY)

# =====================================
# PDF FILE PATH
# =====================================

BOOK_FILE = "i-too-had-a-luv-story.pdf"

# =====================================
# READ THE PDF
# =====================================

def read_pdf(file_path):

    text = ""

    with open(file_path, "rb") as file:

        reader = PyPDF2.PdfReader(file)

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text


# =====================================
# SPLIT THE PDF INTO CHUNKS
# =====================================

def split_text(text, chunk_size=1500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])

    return chunks


# =====================================
# FIND THE MOST RELEVANT CHUNKS
# =====================================

def retrieve_chunks(question, chunks, top_k=3):

    question_words = question.lower().split()

    scores = []

    for chunk in chunks:

        chunk_lower = chunk.lower()

        score = 0

        for word in question_words:

            if word in chunk_lower:
                score += 1

        scores.append(score)

    # Sort chunks based on score
    ranked = sorted(
        zip(scores, chunks),
        key=lambda x: x[0],
        reverse=True
    )

    # Keep only useful chunks
    selected_chunks = []

    for score, chunk in ranked[:top_k]:

        if score > 0:
            selected_chunks.append(chunk)

    return "\n\n".join(selected_chunks)


# =====================================
# LOAD THE PDF
# =====================================

book_content = read_pdf(BOOK_FILE)

# Create chunks
chunks = split_text(book_content)

print("PDF Loaded Successfully!")
print("Type 'exit' to quit.")


# =====================================
# CHAT LOOP
# =====================================

while True:

    question = input("\nYou: ")

    if question.lower() == "exit":

        print("Bot: Goodbye!")
        break

    # Retrieve relevant PDF text
    relevant_text = retrieve_chunks(question, chunks)

    # If nothing is found
    if relevant_text == "":

        print("\nBot: Not in PDF")
        continue

    prompt = f"""
You are an expert PDF assistant.

RULES:
1. Answer ONLY from the PDF content provided.
2. Do NOT use outside knowledge.
3. Do NOT make assumptions.
4. If the answer is not available in the PDF content, reply exactly:
   Not in PDF

PDF CONTENT:
{relevant_text}

QUESTION:
{question}
"""

    try:

        response = client.chat.completions.create(

            model="llama-3.1-8b-instant",

            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2,
            max_tokens=300
        )

        answer = response.choices[0].message.content

        print("\nBot:", answer)

    except Exception as e:

        print("\nError:", e)

PDF Loaded Successfully!
Type 'exit' to quit.
